<p style='text-align: center;'>

## <p style='text-align: center;'> **This is the Notebook for the CIFO Project**

### <p style='text-align: center;'> **Group: C37 Vestigial**




In [ ]:
# Numerical arrays and vectorized operations used for triangle genomes and RMSE calculations.
import numpy as np

# Plotting library used to show final images and convergence curves inside the notebook.
import matplotlib.pyplot as plt

# Random utilities used by the genetic algorithm: initialization, selection, and mutation.
from random import randint, choices, sample, random

# OpenCV is used to load images, draw filled triangles, and save generated outputs.
import cv2

# ABC/abstractmethod are used to define a generic optimization solution template.
from abc import ABC, abstractmethod

# deepcopy prevents offspring/elites from accidentally sharing the same object in memory as parents.
from copy import deepcopy

# OS is used for file path manipulations and directory creation.
import os

# Time is used to measure the runtime of the algorithm and to timestamp output files.
import time

## <center> Simulated Annealing

In [ ]:
class Solution(ABC):
    """Generic abstract class for an optimization solution."""

    def __init__(self, repr=None):
        # If no representation is provided, create a random one using the subclass method.
        if repr is None:
            repr = self.random_initial_representation()

        # Store the representation/genome of the solution.
        self.repr = repr

    def __repr__(self):
        # Defines how the solution is displayed when printed.
        return str(self.repr)

    @abstractmethod
    def fitness(self):
        # Every specific solution must define how quality/fitness is calculated.
        pass

    @abstractmethod
    def random_initial_representation(self):
        # Every specific solution must define how a random starting solution is created.
        pass

In [ ]:
class VermeerSolution(Solution):
    """A candidate painting made from 100 colored triangles."""

    def __init__(self, target_image, repr=None):
        # Store the original image
        self.target_image = target_image

        # Project canvas size
        self.width = 300
        self.height = 400
        self.num_triangles = 100

        super().__init__(repr=repr)

    def random_initial_representation(self):
        random_repr = np.zeros((self.num_triangles, 9), dtype=int)
        for i in range(self.num_triangles):
            cx = randint(0, self.width)
            cy = randint(0, self.height)
            radius = 40
            x1 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y1 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x2 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y2 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x3 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y3 = np.clip(cy + randint(-radius, radius), 0, self.height)
            r, g, b = [randint(0, 255) for _ in range(3)]
            random_repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]
        return random_repr

    def render_canvas(self):
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)
        for gene in self.repr:
            pts = np.array([[gene[0], gene[1]], [gene[2], gene[3]], [gene[4], gene[5]]], np.int32)
            pts = pts.reshape((-1, 1, 2))
            color = (int(gene[6]), int(gene[7]), int(gene[8]))
            cv2.fillPoly(canvas, [pts], color)
        return canvas

    def fitness(self):
        """Calculates RMSE + Geometric Penalty (Shoelace Formula)"""
        generated_image = self.render_canvas()
        target_float = self.target_image.astype(np.float32)
        gen_float = generated_image.astype(np.float32)

        # 1. Erro Visual (RMSE)
        diff = target_float - gen_float
        sq_diff = np.square(diff)
        mse = np.mean(sq_diff)
        rmse = np.sqrt(mse)

        return rmse

    def get_random_neighbor(self):
        """
        Generates a new solution optimized for Simulated Annealing.
        Uses Gaussian distributions and Heuristic color stealing.
        """
        neighbor = VermeerSolution(
            target_image=self.target_image, 
            repr=self.repr.copy()
        )
        
        num_mutations = np.random.randint(1, 4) 
        
        for _ in range(num_mutations):
            i = np.random.randint(0, neighbor.num_triangles)
            mutation_type = random()
            
            if mutation_type < 0.50:
                nudge = np.random.normal(0, 4, 6).astype(int)
                neighbor.repr[i][0:6] += nudge
                neighbor.repr[i][0::2] = np.clip(neighbor.repr[i][0::2], 0, neighbor.width)
                neighbor.repr[i][1::2] = np.clip(neighbor.repr[i][1::2], 0, neighbor.height)
                
            elif mutation_type < 0.85:
                nudge = np.random.normal(0, 5, 3).astype(int)
                neighbor.repr[i][6:9] += nudge
                neighbor.repr[i][6:9] = np.clip(neighbor.repr[i][6:9], 0, 255)
                
            elif mutation_type < 0.95:
                swap_idx = np.random.randint(0, neighbor.num_triangles)
                neighbor.repr[[i, swap_idx]] = neighbor.repr[[swap_idx, i]]
                
            else:
                cx = np.random.randint(0, neighbor.width)
                cy = np.random.randint(0, neighbor.height)
                
                target_color = self.target_image[cy, cx]
                b, g, r = int(target_color[0]), int(target_color[1]), int(target_color[2])
                
                radius = 35
                x1 = np.clip(cx + np.random.randint(-radius, radius), 0, neighbor.width)
                y1 = np.clip(cy + np.random.randint(-radius, radius), 0, neighbor.height)
                x2 = np.clip(cx + np.random.randint(-radius, radius), 0, neighbor.width)
                y2 = np.clip(cy + np.random.randint(-radius, radius), 0, neighbor.height)
                x3 = np.clip(cx + np.random.randint(-radius, radius), 0, neighbor.width)
                y3 = np.clip(cy + np.random.randint(-radius, radius), 0, neighbor.height)
                
                neighbor.repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]
                
        return neighbor

In [ ]:
def simulated_annealing(
    initial_solution: Solution,
    C: float,
    L: int,
    cooling_rate: float,
    max_iter: int = 10,
    verbose: bool = False,
):
    """
    Generic Simulated Annealing implementation kept from class material.

    NOTE: VermeerSolution would need get_random_neighbor() implemented for this to work on the painting problem.
    """
    # Start from the provided initial solution and calculate its fitness.
    current_solution = initial_solution
    current_fitness = current_solution.fitness()

    # Best global solution as SA can accept worse ones
    best_solution = initial_solution
    best_fitness = current_fitness

    C = C
    rmse_history = []  
    total_evals = 0

    for iteration in range(1, max_iter + 1):

        for _ in range(L):
            # Generate a neighbor with a slight mutation
            neighbor = current_solution.get_random_neighbor()
            neighbor_fitness = neighbor.fitness()
            total_evals += 1

            delta = neighbor_fitness - current_fitness   # less than 0, improvement

            if delta <= 0:
                # if the neighbor is better, accept it as current solution
                current_solution = neighbor
                current_fitness = neighbor_fitness
            else:
                # if the neighbor is worse, accept with probability of Boltzmann
                prob = np.exp(-delta / C) if C > 1e-10 else 0.0
                if random() < prob:
                    current_solution = neighbor
                    current_fitness = neighbor_fitness

            # Update global best
            if current_fitness < best_fitness:
                best_fitness = current_fitness
                # Save a copy of the best solution
                best_solution = VermeerSolution(
                    target_image=current_solution.target_image,
                    repr=current_solution.repr.copy(),
                )

            rmse_history.append(current_fitness)

        # Geometric cooling
        C *= cooling_rate

        if verbose and iteration % 500 == 0:
            print(
                f"  Iter {iteration:5d}/{max_iter} | "
                f"C={C:.4f} | "
                f"RMSE atual={current_fitness:.2f} | "
                f"Best={best_fitness:.2f}"
            )

    return best_solution, rmse_history


In [ ]:
# Path to local copies of the target image.
joao_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
goncalo_path = r"C:\CIfO\Project\data\girl_pearl_earing.png"
rafa_path = r"Project\girl_pearl_earing.png"
gui_path = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
# Load the original image using OpenCV.
original_image = cv2.imread(gui_path)

# Fail early with a clear error if the path is wrong or the image cannot be read.
if original_image is None:
    raise FileNotFoundError(f"OpenCV could not find the image at: {gui_path}")

In [ ]:
# 1. Define the target image
target = original_image 

# 2. Optimized Parameters (L * max_iter = 1,600,000 evaluations)
L_param = 100         
max_iter_param = 8000 
C_inicial = 20.0      
cooling_rate_param = 0.9990 

# 3. Initialize the first solution
sa_initial = VermeerSolution(target_image=target)

print(f"Starting Simulated Annealing with {L_param * max_iter_param} evaluations...")
print(f"Initial Temperature: {C_inicial} | Cooling Rate: {cooling_rate_param}")

# 4. Execute the Algorithm (verbose=False is mandatory to prevent freezing!)
best_sa_sol, sa_history = simulated_annealing(
    initial_solution=sa_initial,
    C=C_inicial, 
    L=L_param, 
    cooling_rate=cooling_rate_param, 
    max_iter=max_iter_param, 
    verbose=True
)

# 5. Process data for the convergence plot (track the best-so-far)
best_so_far_sa = []
current_best = sa_history[0]

for score in sa_history:
    if score < current_best:
        current_best = score
    best_so_far_sa.append(current_best)

print(f"Final SA RMSE: {best_sa_sol.fitness():.2f}")

# 6. Display the Convergence Plot
plt.figure(figsize=(10, 5))
plt.plot(best_so_far_sa, label="Simulated Annealing (Optimized)", color="orange")
plt.xlabel("Total Fitness Function Evaluations")
plt.ylabel("RMSE (Error)")
plt.title("Simulated Annealing Convergence")
plt.legend()
plt.grid(True)
plt.show()

# 7. Display the Final Generated Image
plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(best_sa_sol.render_canvas(), cv2.COLOR_BGR2RGB))
plt.title(f"Optimized SA Result (Final RMSE: {best_sa_sol.fitness():.2f})")
plt.axis("off")
plt.show()